# 01 — Exploración de las 5 fuentes de datos

Este notebook carga las 5 fuentes del Modelo 1 y produce las gráficas exploratorias que justifican el diseño del ABM.

**Requisitos:** `abm-enso download` ya corrido (las 5 fuentes en `data/raw/`).

**Contenido:**
1. ONI — índice ENSO histórico (NOAA/CPC)
2. ERA5 — precip, humedad, runoff (Copernicus)
3. SIRH — nivel hidrométrico (IDEAM/Socrata)
4. SIMMA — inventario de movimientos en masa (SGC)
5. Cuencas — zonificación hidrográfica (IDEAM)
6. Síntesis: ¿qué nos dicen los datos sobre el diseño del ABM?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.patches as mpatches

from abm_enso.data import oni, era5, sirh, simma, cuencas
from abm_enso.utils import constants as C
from abm_enso.utils.paths import FIGURES_DIR

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. ONI — Índice ENSO

El ONI es el forzamiento externo del ABM. Valores ≤ −0.5 °C por ≥5 meses consecutivos = La Niña; ≥ +0.5 °C = El Niño.

In [ ]:
df_oni = oni.load()
print(f'Rango temporal: {df_oni.index.min():%Y-%m} → {df_oni.index.max():%Y-%m}')
print(f'{len(df_oni)} observaciones mensuales')
df_oni.describe().T

In [ ]:
# Replica el estilo del Fuente_1.py original
fig, ax = plt.subplots(figsize=(14, 5))
vals = df_oni['oni'].values
fechas = df_oni.index

ax.fill_between(fechas, np.minimum(vals, 0), 0, color='#4A90D9', alpha=0.75, linewidth=0)
ax.fill_between(fechas, np.maximum(vals, 0), 0, color='#C05050', alpha=0.70, linewidth=0)
ax.plot(fechas, vals, color='#1A2332', linewidth=0.7, alpha=0.85)

ax.axhline(C.UMBRAL_NINA, color='#4A90D9', ls='--', lw=1)
ax.axhline(C.UMBRAL_NINO, color='#C05050', ls='--', lw=1)
ax.axhline(0, color='#AAA', lw=0.5)

# Anotar La Niña 2010-11
nina = df_oni.loc['2010-06':'2011-06', 'oni']
if len(nina) > 0:
    idx_min = nina.idxmin()
    ax.annotate(f'La Niña 2010-11\n{nina.min():.1f}°C · 3.2M afectados',
                xy=(idx_min, nina.min()),
                xytext=(idx_min + pd.DateOffset(months=18), nina.min() + 0.5),
                fontsize=10, fontweight='bold',
                arrowprops=dict(arrowstyle='->', lw=1.5, connectionstyle='arc3,rad=-0.25'))

ax.set_ylim(-3.2, 3.2)
ax.set_xlabel('Año'); ax.set_ylabel('Anomalía ONI (°C)')
ax.set_title(f'Índice ONI {df_oni.index.min():%Y}–{df_oni.index.max():%Y}  —  NOAA/CPC',
             fontsize=13, fontweight='bold')
ax.legend(handles=[mpatches.Patch(color='#4A90D9', alpha=0.75, label='La Niña (ONI < -0.5°C)'),
                   mpatches.Patch(color='#C05050', alpha=0.70, label='El Niño (ONI > +0.5°C)')])
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '01_oni_full.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. ERA5 — Precipitación, humedad, escorrentía

Tres variables promediadas sobre Colombia. Las bandas rojas/verdes marcan eventos ENSO de referencia.

In [ ]:
df_era5 = era5.load()
print(f'Rango: {df_era5.index.min():%Y-%m} → {df_era5.index.max():%Y-%m}')
print(f'Columnas: {list(df_era5.columns)}')
df_era5.describe().T

In [ ]:
ENSO_EVENTS = [
    ('1988-06-01', '1989-05-01', '#86EFAC', 0.35),  # Niña
    ('1997-06-01', '1998-05-01', '#FCA5A5', 0.35),  # Niño
    ('2010-06-01', '2011-05-01', '#86EFAC', 0.35),  # Niña
    ('2015-04-01', '2016-05-01', '#FCA5A5', 0.35),  # Niño
]

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

specs = [
    ('precip_mm_mes',     'Precipitación',  'mm/mes',    '#93C5FD', '#1A3A6B'),
    ('runoff_mm_mes',     'Escorrentía',    'mm/mes',    '#5EEAD4', '#0F766E'),
    ('humedad_suelo_pct', 'Humedad suelo',  '% vol.',    '#D6B88F', '#7C2D12'),
]

for ax, (col, titulo, ylab, cbar, cline) in zip(axes, specs):
    ax.bar(df_era5.index, df_era5[col], color=cbar, alpha=0.6, width=30, edgecolor='none', label='Mensual')
    mm12 = df_era5[col].rolling(12, center=True, min_periods=1).mean()
    ax.plot(df_era5.index, mm12, color=cline, lw=1.6, label='Media 12m')
    ax.axhline(df_era5[col].mean(), color='#6B7280', ls='--', lw=1, alpha=0.7)
    
    for ini, fin, color, alpha in ENSO_EVENTS:
        ax.axvspan(pd.Timestamp(ini), pd.Timestamp(fin), color=color, alpha=alpha, zorder=0)
    
    ax.set_title(f'{titulo} — ERA5 Colombia', fontsize=11, fontweight='bold', loc='left')
    ax.set_ylabel(ylab)
    ax.spines[['top','right']].set_visible(False)
    ax.legend(loc='upper left', fontsize=8, ncol=2)

axes[-1].set_xlabel('Año')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_era5_3vars.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. SIRH — Nivel hidrométrico

3 estaciones IDEAM en cuencas andinas distintas. Banda roja = La Niña 2020-23.

In [ ]:
try:
    df_sirh = sirh.load()
    print(f'{len(df_sirh):,} observaciones diarias en {df_sirh["codigoestacion"].nunique()} estaciones')
    print(df_sirh.groupby('codigoestacion').size().to_string())
except FileNotFoundError as e:
    print(f'SIRH no descargado: {e}')
    print('Corre: abm-enso download --solo sirh')
    df_sirh = None

In [ ]:
if df_sirh is not None:
    estaciones = sirh.ESTACIONES_PILOTO
    colores = {'0035077180': '#1A3A6B', '0026157190': '#0E7C7B', '0021167080': '#B45309'}
    
    fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
    for ax, (cod, (nombre, cuenca)) in zip(axes, estaciones.items()):
        sub = df_sirh[df_sirh['codigoestacion'] == cod]
        if len(sub) == 0:
            ax.set_title(f'{cod} — sin datos', loc='left')
            continue
        ax.plot(sub['fecha'], sub['nivel_m'], color=colores[cod], lw=0.6)
        ax.set_ylabel('Nivel (m)')
        ax.set_title(f'{cod} — {nombre}  [{cuenca}]', loc='left', fontweight='bold',
                     color=colores[cod])
        ax.axvspan(pd.Timestamp('2020-08-01'), pd.Timestamp('2023-03-31'),
                   color='#FCA5A5', alpha=0.25, label='La Niña 2020-23')
        ax.legend(loc='upper right', fontsize=8)
        ax.spines[['top','right']].set_visible(False)
    
    fig.suptitle('Nivel hidrométrico IDEAM — 3 cuencas andinas', fontweight='bold', y=1.0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '03_sirh_niveles.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. SIMMA — Inventario de movimientos en masa

Eventos confirmados por el SGC. Analizamos: distribución temporal, por departamento y por tipo.

In [ ]:
df_simma = simma.load()
print(f'{len(df_simma):,} eventos SIMMA')
print(f'Rango: {df_simma["fecha_evento"].min():%Y-%m} → {df_simma["fecha_evento"].max():%Y-%m}')
print(f'\nTipos:\n{df_simma["tipo_movimiento"].value_counts()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: serie temporal
df_mensual = df_simma.set_index('fecha_evento').resample('MS').size()
axes[0].fill_between(df_mensual.index, df_mensual.values, color='#C05050', alpha=0.4, label='Eventos/mes')
mm12 = df_mensual.rolling(12, center=True, min_periods=1).mean()
axes[0].plot(mm12.index, mm12.values, color='#7C2D12', lw=1.5, label='Media 12m')
axes[0].axvspan(pd.Timestamp('2010-06-01'), pd.Timestamp('2011-05-01'),
                color='#86EFAC', alpha=0.3, label='La Niña 2010-11')
axes[0].set_title('Eventos SIMMA por mes', loc='left', fontweight='bold')
axes[0].set_ylabel('Número de eventos')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Panel B: Top 15 departamentos
top_dept = df_simma['departamento'].value_counts().head(15)
axes[1].barh(range(len(top_dept)), top_dept.values[::-1], color='#1A3A6B', alpha=0.8)
axes[1].set_yticks(range(len(top_dept)))
axes[1].set_yticklabels(top_dept.index[::-1], fontsize=9)
axes[1].set_xlabel('Número de eventos')
axes[1].set_title('Top 15 departamentos', loc='left', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_simma_resumen.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Cuencas IDEAM — Geometría espacial

Los polígonos serán los agentes del ABM. Cada polígono recibirá parámetros β₁, θ, κ heterogéneos.

In [ ]:
try:
    gdf = cuencas.load()
    print(f'{len(gdf)} polígonos')
    print(f'CRS: {gdf.crs}')
    print(f'Columnas: {list(gdf.columns)}')
    gdf.head()
except FileNotFoundError as e:
    print(f'Cuencas no descargadas: {e}')
    gdf = None

In [ ]:
if gdf is not None:
    fig, ax = plt.subplots(figsize=(10, 11))
    gdf.plot(ax=ax, facecolor='#E9EFE6', edgecolor='#6B7D6D', lw=0.5)
    
    # Superponer eventos SIMMA si tienen lat/lon
    if 'latitud' in df_simma.columns and df_simma['latitud'].notna().any():
        df_geo = df_simma.dropna(subset=['latitud', 'longitud'])
        ax.scatter(df_geo['longitud'], df_geo['latitud'],
                   s=4, c='#C05050', alpha=0.4, edgecolors='none',
                   label=f'Eventos SIMMA (n={len(df_geo):,})')
    
    ax.set_xlim(-80, -66); ax.set_ylim(-5, 13)
    ax.set_aspect('equal')
    ax.set_title(f'Cuencas hidrográficas — {len(gdf)} polígonos', fontweight='bold')
    ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
    ax.legend(loc='lower left')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '05_cuencas_simma.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Síntesis — ¿qué nos dicen los datos sobre el ABM?

| Pregunta del ABM | Evidencia empírica |
|---|---|
| ¿El ONI es un forzamiento relevante? | ✓ Picos ERA5 precip coinciden con ONI<-0.5 en 1988, 2010, 2020 |
| ¿Las cuencas son heterogéneas? | ✓ 316+ polígonos con áreas y climas diversos |
| ¿Los eventos se concentran en ENSO? | ✓ Visible en el panel A de SIMMA — picos en 2010-11, 2020-22 |
| ¿Hay lag lluvia → nivel río? | ✓ Observable en SIRH: pico 2020-22 desfasado vs ERA5 |
| ¿La distribución es espacial? | ✓ Cordillera andina concentra eventos SIMMA |

**Siguiente paso (Fase 3):** con estos datos limpios, calibramos β₁, θ, κ contra SIMMA vía grid search.